### Imports

In [ ]:
import os, sys

# Go to the repo root (adjust path if needed)
%cd /content/Data-Science-Project

# Make sure repo root is on sys.path
repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.append(repo_root)

print("Using repo_root:", repo_root)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import AutoTokenizer
from tqdm import tqdm

from data.crisismmd import (
    load_crisismmd_annotations,
    build_fusion_dataframe,
    MultimodalCrisisDataset,
    make_eval_transforms,
)
from models.fusion_model import MultimodalFusionNetwork, FusionDatasetWrapper
from training.utils import train_one_epoch, evaluate

### Load CrisisMMD and fusion dataframe

In [ ]:
# Root directory where CrisisMMD_v2.0 is extracted
root = "../CrisisMMD_v2.0"  # adjust if needed

# Load and merge all annotation .tsv files
combined = load_crisismmd_annotations(root)

# Build dataframe with OR-label: label=1 if text or image is informative, else 0
fusion_df = build_fusion_dataframe(combined)

print("Label distribution (0=safe, 1=disaster):")
print(fusion_df["label"].value_counts())

# Train / validation split
train_df, val_df = train_test_split(fusion_df, test_size=0.2, random_state=42)

len(train_df), len(val_df)

### Datasets and loaders

In [ ]:
# Paths to previously trained models
text_model_dir  = "../checkpoints/text_branch"       # HF folder saved by textbranch
vision_weights  = "../checkpoints/vision_brain.pth"  # .pth saved by visionbranch

# Tokenizer for text
tokenizer = AutoTokenizer.from_pretrained(text_model_dir)

# Image transforms (evaluation-style; encoders are frozen)
img_tf = make_eval_transforms()

# Base multimodal datasets: each item is a dict with
#   'input_ids', 'attention_mask', 'image', 'label'
base_train_ds = MultimodalCrisisDataset(
    df=train_df,
    root_dir=root,
    tokenizer=tokenizer,
    image_transform=img_tf,
)
base_val_ds = MultimodalCrisisDataset(
    df=val_df,
    root_dir=root,
    tokenizer=tokenizer,
    image_transform=img_tf,
)

train_ds = FusionDatasetWrapper(base_train_ds)
val_ds   = FusionDatasetWrapper(base_val_ds)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

# Quick sanity check
(inputs_batch, labels_batch) = next(iter(train_loader))
print("Shapes:")
print("  input_ids:",      inputs_batch[0].shape)
print("  attention_mask:", inputs_batch[1].shape)
print("  image:",          inputs_batch[2].shape)
print("  labels:",         labels_batch.shape)

### Build fusion model, criterion and optimizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MultimodalFusionNetwork(
    text_model_dir=text_model_dir,
    vision_weights_path=vision_weights,
    device=device,
).to(device)

# Compute class weights for imbalance (0: safe, 1: disaster)
count_safe_0     = (fusion_df["label"] == 0).sum()
count_disaster_1 = (fusion_df["label"] == 1).sum()
total = count_safe_0 + count_disaster_1

w0 = total / count_safe_0
w1 = total / count_disaster_1

class_weights = torch.tensor([w0, w1], dtype=torch.float32).to(device)
print("Class weights:", class_weights.cpu().numpy())

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Only train the fusion head; encoders are frozen in MultimodalFusionNetwork
optimizer = torch.optim.Adam(model.fusion_classifier.parameters(), lr=1e-3)

EPOCHS = 10

train_loss_hist = []
train_acc_hist  = []
val_loss_hist   = []
val_acc_hist    = []

### Training loop

In [ ]:
print("Starting Late Fusion Training...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
    )

    val_loss, val_acc, _, _ = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
    )

    train_loss_hist.append(train_loss)
    train_acc_hist.append(train_acc)
    val_loss_hist.append(val_loss)
    val_acc_hist.append(val_acc)

    print(
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )

print("\nMultimodal Fusion Training complete.")

### Save fusion weights

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)
fusion_path = "../checkpoints/fusion_brain.pth"
torch.save(model.state_dict(), fusion_path)
print(f"Saved fusion model to {fusion_path}")

### Plot train/val curves

In [ ]:
epochs = np.arange(1, EPOCHS + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
ax[0].plot(epochs, train_loss_hist, marker="o", label="Train")
ax[0].plot(epochs, val_loss_hist,   marker="o", label="Val")
ax[0].set_title("Fusion Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].legend()

# Accuracy curves
ax[1].plot(epochs, train_acc_hist, marker="o", label="Train")
ax[1].plot(epochs, val_acc_hist,   marker="o", label="Val")
ax[1].set_title("Fusion Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].legend()

plt.tight_layout()
plt.show()

### Evaluation and confusion matrix

In [ ]:
val_loss, val_acc, val_preds, val_true = evaluate(
    model=model,
    loader=val_loader,
    criterion=criterion,
    device=device,
)

val_f1 = f1_score(val_true, val_preds, average="binary")

print("\n--- ULTIMATE MULTIMODAL FUSION VALIDATION RESULTS ---")
print(f"Val Accuracy: {val_acc * 100:.2f}%")
print(f"Val F1 Score: {val_f1:.4f}")

cm = confusion_matrix(val_true, val_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Safe (0)", "Disaster (1)"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Purples", ax=ax, colorbar=False)
plt.title("Late Fusion Model - Confusion Matrix (Validation)", fontsize=14)
plt.grid(False)
plt.show()